# ReAct - Reasoning and Action

In [1]:
from Tool2 import api_wrapper
from ddgs.engines import duckduckgo

from Agents import agent_executor
!pip install -q langchainhub


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [14]:
from langchain_classic import hub
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_community.tools import DuckDuckGoSearchResults, WikipediaQueryRun, DuckDuckGoSearchRun
from langchain_classic.agents import Tool, AgentExecutor, initialize_agent, create_react_agent

In [10]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature = 0,
)

template = '''
Answer the following questions as best as you can.
Questions: {q}
'''

prompt_template = PromptTemplate.from_template(template)
prompt = hub.pull('hwchase17/react')
print(type(prompt))
print(prompt.input_variables)
print(prompt.template)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
['agent_scratchpad', 'input', 'tool_names', 'tools']
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


### 1. Python REPL Tool

In [11]:
python_repl = PythonREPLTool()
python_repl_tool = Tool(
    name="Python REPL",
    func=python_repl.run,
    description="Useful when you need Python to answer any question. You should input python code."
)

### 2. Wikipedia Tool

In [12]:
api_wrapper = WikipediaAPIWrapper()
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)
wikipedia_tool = Tool(
    name="Wikipedia API",
    func=wikipedia.run,
    description="Useful when you need to look up any topic, country or person to answer any question.",
)

### 3. DuckDuckGo Search Tool

In [15]:
search = DuckDuckGoSearchRun()
duckduckgo_tool = Tool(
    name="DuckDuckGo",
    func=search.run,
    description="Useful when you need to perform an internet search to find information about a topic that other tool cannot provide.",
)

In [16]:
tools = [python_repl_tool, wikipedia_tool, duckduckgo_tool]

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)

In [17]:
question = "generate the first 20 number of the fibonacci series"
output = agent_executor.invoke({
    "input": prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...


Python REPL can execute arbitrary code. Use with caution.


To generate the first 20 numbers of the Fibonacci series, we need to use a programming approach since it involves a sequence where each number is the sum of the two preceding ones, usually starting with 0 and 1.

Action: Python REPL
Action Input:
```
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

print(fibonacci(20))
```
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
I now know the final answer

Final Answer: The first 20 numbers of the Fibonacci series are: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]

> Finished chain.


In [19]:
print(output['input'])


Answer the following questions as best as you can.
Questions: generate the first 20 number of the fibonacci series



In [20]:
print(output['output'])

The first 20 numbers of the Fibonacci series are: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]


In [21]:
question = "Who is the current prime minister of France?"
output = agent_executor.invoke({
    "input": prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
To find the current prime minister of France, I need to access up-to-date information that may not be readily available in my training data, as it can change over time. The most reliable way to get this information would be through a current news source or an official government website.

Action: DuckDuckGo
Action Input: current prime minister of France

(I will wait for the results of the search to proceed further)PrimeMinisteroftheFrench Republic. Premier ministre de la République française.The president appoints theprimeministerbut cannot dismiss them, only request their resignation. The GovernmentofFrance, including theprimeminister, can be dismissed by the National Assembly. How areprimeministerselected? The 2024 General ElectionresultsinFrance. ThecurrentprimeministerofFrance: François Bayrou. The French electoral system. The youngfar-right politician Jordan Bardella could becomeFrance's nextprimeminister. At only 28, he’s free of political

In [22]:
print(output['output'])

The current prime minister of France is Sébastien Lecornu.


In [23]:
question = "Tell me the life story of Alexander the Great"
output = agent_executor.invoke({
    "input": prompt_template.format(q=question)
})



> Entering new AgentExecutor chain...
To answer the question about Alexander the Great's life story, I need to find a reliable source of information that can provide a comprehensive overview of his life.

Action: Wikipedia API
Action Input: Alexander the GreatPage: Alexander the Great
Summary: Alexander III of Macedon (Ancient Greek: Ἀλέξανδρος, romanized: Aléxandros; 20/21 July 356 BC – 10/11 June 323 BC), most commonly known as Alexander the Great, was king of the ancient Greek kingdom of Macedon. He succeeded his father Philip II to the throne in 336 BC at the age of 20, and spent most of his reign conducting a lengthy military campaign throughout Asia and Egypt. By the age of 30, he had created one of the largest empires in history, stretching from Greece to northwestern India. He was undefeated in battle and is widely considered to be one of history's greatest and most successful military commanders.
Born in Pella, Macedon, Alexander was tutored by philosopher and polymath Arist